In [1]:
# ✅ ENV SETUP
import os
from dotenv import load_dotenv

load_dotenv(".env")

FIGMA_TOKEN = os.getenv("FIGMA_TOKEN")
FILE_ID_1 = os.getenv("FIGMA_DOCUMENT_ID1")
FILE_ID_2 = os.getenv("FIGMA_DOCUMENT_ID2")
SAS_URL = os.getenv("SAS_URL")

pages_1 = ["Module 1", "Module 2", "Module 3", "Module 4"]
pages_2 = ["Module 5", "Module 6", "Module 7", "Module 8"]

required = ["FIGMA_TOKEN", "FILE_ID_1", "FILE_ID_2", "SAS_URL"]
missing = [v for v in required if not globals()[v]]
if missing:
    raise EnvironmentError(f"❌ Missing env vars: {', '.join(missing)}")

print("✅ Environment loaded.")




✅ Environment loaded.


In [2]:
# ✅ UTILS
import io
from PIL import Image
import time
from contextlib import contextmanager

def format_blob_name(folder: str, name: str) -> str:
    return f"{folder}/{name.replace(':', '_').replace(' ', '_')}.webp"

def convert_image_bytes_to_webp(image_bytes: bytes, resize_to=1080, quality=95) -> bytes:
    with Image.open(io.BytesIO(image_bytes)) as img:
        img = img.convert("RGB")
        if img.width > resize_to or img.height > resize_to:
            img.thumbnail((resize_to, resize_to), Image.Resampling.LANCZOS)
        buffer = io.BytesIO()
        img.save(buffer, format="WEBP", quality=quality)
        return buffer.getvalue()

@contextmanager
def timed(label):
    t0 = time.time()
    yield
    print(f"⏱️ {label} took {time.time() - t0:.2f}s")

In [3]:
# ✅ AZURE STORAGE
import ssl
from azure.storage.blob.aio import ContainerClient
from azure.storage.blob import ContentSettings
from azure.core.pipeline.transport import AioHttpTransport

USE_INSECURE_SSL = False  # Toggle if debugging in insecure environments

def get_container_client():
    if USE_INSECURE_SSL:
        ssl_context = ssl.create_default_context()
        ssl_context.check_hostname = False
        ssl_context.verify_mode = ssl.CERT_NONE
        transport = AioHttpTransport(ssl_context=ssl_context)
        return ContainerClient.from_container_url(SAS_URL, transport=transport)
    return ContainerClient.from_container_url(SAS_URL)

async def get_existing_blob_names(prefix: str = "") -> set[str]:
    blob_names = set()
    async with get_container_client() as client:
        async for blob in client.list_blobs(name_starts_with=prefix):
            blob_names.add(blob.name)
    print(f"📦 {len(blob_names)} blobs found with prefix '{prefix}'")
    return blob_names

async def upload_image_blob(blob_name: str, data: bytes) -> str:
    async with get_container_client() as client:
        blob = client.get_blob_client(blob_name)
        await blob.upload_blob(
            data=data,
            overwrite=True,
            content_settings=ContentSettings(content_type="image/webp"),
        )
        return blob.url.split("?")[0]


In [4]:
# ✅ FIGMA API + IMAGE REF LOGIC
import requests
import json
import os

REFS_PATH = "figma_image_refs.json"

def fetch_figma_file(file_id: str) -> dict:
    res = requests.get(
        f"https://api.figma.com/v1/files/{file_id}",
        headers={"X-Figma-Token": FIGMA_TOKEN}
    )
    res.raise_for_status()
    return res.json()

def fetch_image_url_single(file_id: str, node_id: str, format="png") -> str | None:
    try:
        res = requests.get(
            f"https://api.figma.com/v1/images/{file_id}",
            headers={"X-Figma-Token": FIGMA_TOKEN},
            params={"ids": node_id, "format": format},
            timeout=30,
        )
        res.raise_for_status()
        return res.json().get("images", {}).get(node_id)
    except Exception as e:
        print(f"❌ Error fetching URL for {node_id}: {e}")
        return None


def load_image_refs() -> dict[str, str]:
    if os.path.exists(REFS_PATH):
        with open(REFS_PATH, "r") as f:
            return json.load(f)
    return {}

def save_image_refs(refs: dict[str, str]):
    with open(REFS_PATH, "w") as f:
        json.dump(refs, f, indent=2)

def extract_image_refs(node: dict, image_refs: dict[str, str]):
    """
    Recursively extract node ID and its imageRef (Figma's internal hash) from rectangles with IMAGE fill.
    """
    if node.get("type") == "RECTANGLE":
        fills = node.get("fills", [])
        for fill in fills:
            if fill.get("type") == "IMAGE":
                image_ref = fill.get("imageRef")
                if image_ref:
                    image_refs[node["id"]] = image_ref
                break
    for child in node.get("children", []):
        extract_image_refs(child, image_refs)

def get_images_to_upload(
    id_to_blobname: dict[str, str],
    image_refs: dict[str, str],
    previous_refs: dict[str, str],
    existing_blobs: set[str]
) -> dict[str, str]:
    to_upload = {}
    normalized_existing = {b.lower() for b in existing_blobs}

    for node_id, blob_name in id_to_blobname.items():
        current_ref = image_refs.get(node_id)
        previous_ref = previous_refs.get(node_id)
        blob_name_norm = blob_name.lower()

        reason = None
        
        if current_ref != previous_ref:
            reason = "imageRef changed"
        elif blob_name_norm not in normalized_existing:
            reason = "new blob"
        else:
            print(f"✅ SKIP: {blob_name} | unchanged | imageRef: {current_ref}")
            continue

        print(f"📤 UPLOAD: {blob_name} | reason: {reason} | currentRef: {current_ref} | previousRef: {previous_ref}")
        to_upload[node_id] = blob_name

    return to_upload



In [5]:
import aiohttp
from PIL import Image
import io

def convert_image_bytes_to_webp(image_bytes: bytes, resize_to=1080, quality=95) -> bytes:
    with Image.open(io.BytesIO(image_bytes)) as img:
        img = img.convert("RGB")
        if img.width > resize_to or img.height > resize_to:
            img.thumbnail((resize_to, resize_to), Image.Resampling.LANCZOS)
        buffer = io.BytesIO()
        img.save(buffer, format="WEBP", quality=quality)
        return buffer.getvalue()

async def fetch_image_content(url: str) -> bytes | None:
    try:
        async with aiohttp.ClientSession() as session:
            async with session.get(url, timeout=30) as resp:
                if resp.status == 200:
                    return await resp.read()
                print(f"⚠️ Failed to fetch {url} (status {resp.status})")
    except Exception as e:
        print(f"❌ Error fetching image: {e}")
    return None

In [6]:
import json
import os

async def upload_figma_images(
    file_id: str,
    id_to_blobname: dict[str, str],
    image_refs: dict[str, str],       # Figma's current imageRef map
    existing_blobs: set[str],         # Already in Azure
    cache_path: str,                  # Where to load and write cache
    overwrite: bool = False,
):
    # Load existing cache dict from disk
    try:
        if os.path.exists(cache_path):
            with open(cache_path, "r") as f:
                previous_refs = json.load(f)
        else:
            previous_refs = {}
    except Exception as e:
        print(f"❌ Failed to load cache from {cache_path}: {e}")
        previous_refs = {}

    total = len(id_to_blobname)
    to_upload = {}

    for node_id, blob_name in id_to_blobname.items():
        current_ref = image_refs.get(node_id)
        previous_ref = previous_refs.get(node_id)

        should_upload = False
        reason = ""

        if overwrite:
            should_upload = True
            reason = "overwrite=True"
        elif blob_name not in existing_blobs:
            should_upload = True
            reason = "missing blob"
        elif current_ref != previous_ref:
            should_upload = True
            reason = "imageRef changed"

        if should_upload:
            to_upload[node_id] = blob_name

    unchanged = total - len(to_upload)
    print(f"🧮 {total} total images | {len(to_upload)} to upload | {unchanged} unchanged")

    if not to_upload:
        print("✅ All images are already up to date.")
        return

    # Upload images
    uploaded = 0
    for i, (node_id, blob_name) in enumerate(to_upload.items(), 1):
        url = fetch_image_url_single(file_id, node_id)
        if not url:
            print(f"⚠️  [{i}] No URL for node {node_id}")
            continue

        image_bytes = await fetch_image_content(url)
        if not image_bytes:
            print(f"❌ [{i}] Failed to fetch image content for {node_id}")
            continue

        webp = convert_image_bytes_to_webp(image_bytes)

        try:
            await upload_image_blob(blob_name, webp)
            print(f"✅ [{i}] Uploaded {blob_name}")
            uploaded += 1

            # Update cache on disk immediately
            previous_refs[node_id] = image_refs[node_id]
            try:
                with open(cache_path, "w") as f:
                    json.dump(previous_refs, f, indent=2)
            except Exception as e:
                print(f"❌ Failed to write updated cache after {blob_name}: {e}")

        except Exception as e:
            print(f"❌ [{i}] Failed to upload {blob_name}: {e}")

    print(f"📊 Upload complete | Uploaded: {uploaded} | Skipped: {unchanged}")


In [7]:
def format_path(name: str) -> str:
    return name.replace(" ", "_").strip()


def assign_parents(node: dict, parent: dict = None):
    node["_parent"] = parent
    for child in node.get("children", []) or []:
        assign_parents(child, node)


import re

def extract_lesson_id_from_section_hierarchy(section_name, subsection_name, page_name) -> str | None:
    try:
        chapter_number = re.search(r"Chapter\s+(\d+)", section_name).group(1)
        lesson_number = re.search(r"Lesson\s+(\d+)", subsection_name).group(1)
        module_number = re.search(r"Module\s+(\d+)", page_name).group(1)
        lesson_id = f"{module_number}{chapter_number}{lesson_number}"
        return lesson_id
    except Exception:
        return None



def extract_infographic_images(node: dict, page_name: str) -> list:
    scanned = []
    if node.get("type") == "FRAME" and node.get("name", "").strip().lower() == "infographic":
        for child in node.get("children", []):
            if child.get("type") == "RECTANGLE":
                fills = child.get("fills", [])
                if fills and fills[0].get("type") == "IMAGE":
                    image_ref = fills[0].get("imageRef")
                    if image_ref:
                        scanned.append({
                            "node_id": child["id"],
                            "blob_name": format_blob_name(f"Modules/{format_path(page_name)}", child["id"]),
                            "image_ref": image_ref,
                            "type": "infographic"
                        })
                        print(f"✅ Found infographic image RECTANGLE: {child['id']}")
                        return scanned

        for child in node.get("children", []):
            if child.get("type") in {"GROUP", "INSTANCE", "FRAME"} and child.get("name", "").strip().lower() != "light template":
                fills = child.get("fills", [])
                image_ref = fills[0]["imageRef"] if fills and "imageRef" in fills[0] else None
                if image_ref:
                    print(f"⚠️ Using fallback infographic container: {child['id']}")
                    scanned.append({
                        "node_id": child["id"],
                        "blob_name": format_blob_name(f"Modules/{format_path(page_name)}", child["id"]),
                        "image_ref": image_ref,
                        "type": "infographic"
                    })
                    return scanned
    return []

    
def extract_lesson_cover(node, structure_json, section_name, subsection_name, page_name):
    def find_image_rect(n):
        if n.get("type") == "RECTANGLE":
            for fill in n.get("fills", []):
                if fill.get("type") == "IMAGE" and "imageRef" in fill:
                    return n["id"], fill["imageRef"]
        children = n.get("children")
        if isinstance(children, list):  # ✅ Only iterate if children is a list
            for child in children:
                result = find_image_rect(child)
                if result:
                    return result
        return None

    result = find_image_rect(node)
    if result is None:
        return None

    image_id, image_ref = result

    # Extract module/chapter/lesson ID
    module_match = re.search(r"Module (\d+)", page_name)
    chapter_match = re.search(r"Chapter (\d+)", section_name or "")
    lesson_match = re.search(r"Lesson (\d+)", subsection_name or "")
    if not (module_match and chapter_match and lesson_match):
        return None

    lesson_id = f"{module_match.group(1)}.{chapter_match.group(1)}.{lesson_match.group(1)}"

    blob_name = f"Lessons/frame/lesson{lesson_id.replace('.', '')}_frame.webp"
    return {
        "node_id": node["id"],
        "blob_name": blob_name,
        "image_ref": image_ref,
        "type": "lesson_frame"
    }


def extract_generic_image(node: dict, page_name: str) -> dict | None:
    if node.get("type") != "RECTANGLE":
        return None

    fills = node.get("fills", [])
    for fill in fills:
        if fill.get("type") == "IMAGE" and "imageRef" in fill:
            return {
                "node_id": node["id"],
                "blob_name": format_blob_name(f"Modules/{format_path(page_name)}", node["id"]),
                "image_ref": fill["imageRef"],
                "type": "generic"
            }
    return None


def format_path(name: str) -> str:
    return name.replace(" ", "_").strip()


def assign_parents(node: dict, parent: dict = None):
    node["_parent"] = parent
    for child in node.get("children", []) or []:
        assign_parents(child, node)


import re

def extract_lesson_id_from_section_hierarchy(section_name, subsection_name, page_name) -> str | None:
    try:
        chapter_number = re.search(r"Chapter\s+(\d+)", section_name).group(1)
        lesson_number = re.search(r"Lesson\s+(\d+)", subsection_name).group(1)
        module_number = re.search(r"Module\s+(\d+)", page_name).group(1)
        lesson_id = f"{module_number}{chapter_number}{lesson_number}"
        return lesson_id
    except Exception:
        return None


def extract_infographic_images(node: dict, page_name: str) -> list:
    """
    Export any FRAME named 'infographic' as a full-frame image.
    Uses the frame's node ID for blob naming, consistent with other image exports.
    """
    scanned = []

    if node.get("type") == "FRAME" and node.get("name", "").strip().lower() == "infographic":
        blob_name = format_blob_name(f"Modules/{format_path(page_name)}", node["id"])
        #print(f"✅ Found infographic frame: {node['id']} → {blob_name}")

        scanned.append({
            "node_id": node["id"],
            "blob_name": blob_name,
            "image_ref": node["id"],  # use frame ID directly for export
            "type": "infographic"
        })

    # Recurse to check for nested infographic frames
    for child in node.get("children", []) or []:
        scanned.extend(extract_infographic_images(child, page_name))

    return scanned




def extract_lesson_cover(node, structure_json, section_name, subsection_name, page_name):
    def find_image_rect(n):
        if n.get("type") == "RECTANGLE":
            for fill in n.get("fills", []):
                if fill.get("type") == "IMAGE" and "imageRef" in fill:
                    return n["id"], fill["imageRef"]
        children = n.get("children")
        if isinstance(children, list):  # ✅ Only iterate if children is a list
            for child in children:
                result = find_image_rect(child)
                if result:
                    return result
        return None

    result = find_image_rect(node)
    if result is None:
        return None

    image_id, image_ref = result

    # Extract module/chapter/lesson ID
    module_match = re.search(r"Module (\d+)", page_name)
    chapter_match = re.search(r"Chapter (\d+)", section_name or "")
    lesson_match = re.search(r"Lesson (\d+)", subsection_name or "")
    if not (module_match and chapter_match and lesson_match):
        return None

    lesson_id = f"{module_match.group(1)}.{chapter_match.group(1)}.{lesson_match.group(1)}"

    blob_name = f"Lessons/frame/lesson{lesson_id.replace('.', '')}_frame.webp"
    return {
        "node_id": node["id"],
        "blob_name": blob_name,
        "image_ref": image_ref,
        "type": "lesson_frame"
    }


def extract_generic_image(node: dict, page_name: str) -> dict | None:
    fills = node.get("fills", [])
    for fill in fills:
        if fill.get("type") == "IMAGE" and "imageRef" in fill:
            return {
                "node_id": node["id"],
                "blob_name": format_blob_name(f"Modules/{format_path(page_name)}", node["id"]),
                "image_ref": fill["imageRef"],
                "type": "generic"
            }
    return None
    
def scan_figma_page(page, structure_json):
    scanned = {}
    assign_parents(page)

    type_counts = {
        "generic": 0,
        "lesson_frame": 0,
        "infographic": 0
    }

    def walk(node, section_name=None, subsection_name=None):
        node_type = node.get("type")
        node_name = node.get("name", "")
        children = node.get("children") or []

        # === INFOS: Extract infographic images
        if node_type == "FRAME" and node_name.strip().lower() == "infographic":
            for result in extract_infographic_images(node, page["name"]):
                scanned[result["node_id"]] = result
                type_counts["infographic"] += 1
            # Keep walking in case there are additional nested items

        # === RECTANGLE with IMAGE fill → generic image
        if node_type == "RECTANGLE":
            result = extract_generic_image(node, page["name"])
            if result:
                scanned[result["node_id"]] = result
                type_counts["generic"] += 1

        # === SECTION → Lesson cover
        if node_type == "SECTION":
            for child in children:
                if child.get("type") == "SECTION":
                    # Subsection inside a section
                    for sub_child in child.get("children") or []:
                        if sub_child.get("type") == "FRAME" and sub_child.get("name") == "lesson_cover":
                            result = extract_lesson_cover(
                                sub_child,
                                structure_json,
                                section_name=node_name,
                                subsection_name=child.get("name", ""),
                                page_name=page["name"]
                            )
                            if result:
                                scanned[result["node_id"]] = result
                                type_counts["lesson_frame"] += 1

        # === Walk all children recursively
        for child in children:
            walk(child, section_name=section_name, subsection_name=subsection_name)

    walk(page)

    total = sum(type_counts.values())
    print(f"🔍 Found {total} image nodes → "
          f"{type_counts['generic']} generic | "
          f"{type_counts['lesson_frame']} lesson covers | "
          f"{type_counts['infographic']} infographics")

    return scanned


In [8]:
import json
from pathlib import Path

# === CONFIG ===
structure_path = "../03_Outputs/SEA_Modules/en/module_structure.json"
figma_path_1 = "../02_Inputs/figma_jsons/figma_document1.json"
figma_path_2 = "../02_Inputs/figma_jsons/figma_document2.json"
cache_path = "image_refs_cache.json"

print("📦 Fetching existing Azure blobs...")
existing_blobs = await get_existing_blob_names()

print("🔄 Loading cached image references...")
# Load only for the outer cache final save — not passed to upload_figma_images
updated_refs = load_image_refs()

# === Load structure and figma files ===
with open(structure_path) as f:
    structure_json = json.load(f)
with open(figma_path_1) as f:
    figma_data_1 = json.load(f)
with open(figma_path_2) as f:
    figma_data_2 = json.load(f)

# === Prepare file-page mapping ===
file_page_sets = [
    (FILE_ID_1, figma_data_1["children"], pages_1),
    (FILE_ID_2, figma_data_2["children"], pages_2)
]

# === Loop through files and pages ===
for file_id, figma_pages, page_names in file_page_sets:
    for page_name in page_names:
        page = next((p for p in figma_pages if p["name"] == page_name), None)
        if not page:
            print(f"⚠️ Page not found: {page_name}")
            continue

        print(f"\n📄 Processing page: {page_name}")
        scanned = scan_figma_page(page, structure_json)

        if not scanned:
            print(f"⚠️ No images found on page {page_name}")
            continue

        id_to_blobname = {
            node_id: info["blob_name"].replace(" ", "_")
            for node_id, info in scanned.items()
        }

        fetched_refs = {
            node_id: info["image_ref"]
            for node_id, info in scanned.items()
            if "image_ref" in info
        }

        await upload_figma_images(
            file_id=file_id,
            id_to_blobname=id_to_blobname,
            image_refs=fetched_refs,
            existing_blobs=existing_blobs,
            cache_path=cache_path,
            overwrite=False,
        )

print("✅ Upload pipeline complete.")


📦 Fetching existing Azure blobs...
📦 4501 blobs found with prefix ''
🔄 Loading cached image references...

📄 Processing page: Module 1
🔍 Found 218 image nodes → 193 generic | 14 lesson covers | 11 infographics
🧮 218 total images | 1 to upload | 217 unchanged
✅ [1] Uploaded Modules/Module_1/4580_140.webp
📊 Upload complete | Uploaded: 1 | Skipped: 217

📄 Processing page: Module 2
🔍 Found 286 image nodes → 251 generic | 12 lesson covers | 23 infographics
🧮 286 total images | 0 to upload | 286 unchanged
✅ All images are already up to date.

📄 Processing page: Module 3
🔍 Found 198 image nodes → 175 generic | 12 lesson covers | 11 infographics
🧮 198 total images | 0 to upload | 198 unchanged
✅ All images are already up to date.

📄 Processing page: Module 4
🔍 Found 267 image nodes → 226 generic | 15 lesson covers | 26 infographics
🧮 267 total images | 0 to upload | 267 unchanged
✅ All images are already up to date.

📄 Processing page: Module 5
🔍 Found 200 image nodes → 163 generic | 10 lesson